## 개별 노드(에이전트) 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.25 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성
- `2026.03.25` : 2-Phase 그래프 구조 반영 (Phase 1: resource_estimation + similar_project → Phase 2: scoring + risk_analysis), 노드 팩토리 시그니처 업데이트, phase1_sync 검증 추가, risk_interdependencies 출력 검증, 서버사이드 재계산 검증 추가

**사전 요구사항:**
- PostgreSQL 실행 (`make docker-up && make migrate && make seed`)
- `.env` 파일에 API 키 설정 (OpenAI/Anthropic, Pinecone)
- 노드 실행 순서: deal_structuring → Phase 1 (resource_estimation, similar_project) → Phase 2 (scoring, risk_analysis) → final_verdict

In [ ]:
import sys

sys.path.insert(0, "../../")

from dotenv import load_dotenv

load_dotenv()

### 1. 환경 설정 및 벡터 스토어 초기화

In [ ]:
import json
import uuid

from backend.app.agent.nodes import (
    make_deal_structuring_node,
    make_final_verdict_node,
    make_resource_estimation_node,
    make_risk_analysis_node,
    make_scoring_node,
    make_similar_project_node,
)

from backend.app.agent.graph import hold_verdict_node
from backend.app.db.vector_store import CompanyContextStore, ProjectHistoryStore

# 벡터 스토어 인스턴스 생성
context_store = CompanyContextStore()
project_store = ProjectHistoryStore()

print("벡터 스토어 초기화 완료")

### 2. 샘플 딜 입력 준비

In [ ]:
# 테스트용 deal_id 생성
test_deal_id = str(uuid.uuid4())
print(f"Test Deal ID: {test_deal_id}")

# 정상 딜 입력 — [딜 기본 정보] + [상세 내용] 이중 섹션 형식
sample_deal_input = """[딜 기본 정보]
고객사: 메가테크
고객 규모: 대기업
산업군: 제조업
프로젝트명: 공장 설비 예지보전 AI 시스템 개발
예상 금액: 20000만원
금액 단위: 만원
기간: 5개월

[상세 내용]
## 프로젝트 배경
기존 정기 점검 체계에서 AI 기반 예측 유지보수로 전환하여 설비 가동률 향상 및 유지보수 비용 절감을 목표로 함.

## 기술 요구사항
- Python, TensorFlow 기반 시계열 예측 모델
- IoT 센서 데이터 실시간 수집/처리 파이프라인
- 대시보드 (React 기반)
- MLOps: 모델 재학습 자동화 파이프라인

## 보안 요구사항
- 공장 내 보안 네트워크 환경에서만 운영, 온프레미스 배포 필수

## 결제 조건
착수금 30%, 중간 40%, 완료 30%

## 이해관계자
- 발주측 PM: 설비관리팀 김부장
- 의사결정권자: 제조본부장"""

# 초기 state 구성
state = {
    "deal_id": test_deal_id,
    "deal_input": sample_deal_input,
}
print(f"초기 state keys: {list(state.keys())}")

### 3. deal_structuring 노드

In [ ]:
deal_structuring = make_deal_structuring_node(context_store)
result = await deal_structuring(state)

print("=== deal_structuring 결과 ===")
print(f"status: {result.get('status')}")
print("structured_deal:")
print(json.dumps(result.get("structured_deal", {}), ensure_ascii=False, indent=2))

In [ ]:
# state 누적
state.update(result)

# missing_fields 확인 (Hold 분기 판단 기준 — _CRITICAL_FIELDS만 카운트)
critical_fields = {"customer_name", "customer_industry", "project_overview", "tech_requirements", "expected_amount", "duration_months"}
raw_missing = state.get("structured_deal", {}).get("missing_fields", [])
critical_missing = [f for f in raw_missing if f in critical_fields]
print(f"Missing fields (raw): {raw_missing}")
print(f"Missing fields (critical only): {critical_missing} (count: {len(critical_missing)})")
print(f"Hold 여부: {'Hold (≥3 critical missing)' if len(critical_missing) >= 3 else 'Continue'}")
print(f"amount_unit: {state.get('structured_deal', {}).get('amount_unit', 'N/A')}")

### 4. hold_verdict 분기 테스트

In [ ]:
# missing_fields가 3개 이상인 경우를 시뮬레이션
hold_state = {
    "deal_id": test_deal_id,
    "deal_input": "AI 프로젝트를 하고 싶습니다.",  # 정보 부족
    "structured_deal": {
        "customer_name": None,
        "customer_size": None,
        "customer_industry": None,
        "project_summary": "AI 프로젝트",
        "tech_requirements": [],
        "expected_amount": None,
        "duration_months": None,
        "payment_terms": None,
        "special_notes": None,
        "missing_fields": ["customer_name", "customer_size", "customer_industry", "expected_amount", "duration_months"],
    },
}

hold_result = hold_verdict_node(hold_state)

print("=== hold_verdict 결과 ===")
print(f"verdict: {hold_result['verdict']}")
print(f"total_score: {hold_result['total_score']}")
print(f"final_report:\n{hold_result['final_report']}")

### 5. Phase 1: resource_estimation 노드

Phase 1에서 similar_project와 병렬 실행됩니다.  
**시그니처 변경**: `make_resource_estimation_node(project_store, context_store)` — context_store 추가

In [ ]:
# 시그니처 변경: (project_store) → (project_store, context_store)
resource_estimation = make_resource_estimation_node(project_store, context_store)
resource_result = await resource_estimation(state)

print("=== resource_estimation 결과 ===")
print(json.dumps(resource_result.get("resource_estimate", {}), ensure_ascii=False, indent=2))

# 출력 스키마 상세 검증
re = resource_result.get("resource_estimate", {})
print("\n=== 스키마 검증 ===")
print(f"work_breakdown: {len(re.get('work_breakdown', []))}건")
print(f"phases: {len(re.get('phases', []))}건")
print(f"team_composition: {len(re.get('team_composition', []))}건")
print(f"duration_months: {re.get('duration_months')}")
print(f"risk_buffer_ratio: {re.get('risk_buffer_ratio')}")
print(f"duration_with_buffer: {re.get('duration_with_buffer')}")
if "profitability" in re:
    prof = re["profitability"]
    print(f"profitability.expected_margin: {prof.get('expected_margin')}")
    print(f"profitability.margin_assessment: {prof.get('margin_assessment')}")

In [ ]:
# Phase 1 결과 누적 — resource_estimate
state.update(resource_result)
print(f"현재 state keys: {list(state.keys())}")

### 6. Phase 1: similar_project 노드

Phase 1에서 resource_estimation과 병렬 실행됩니다.  
**시그니처 변경**: `make_similar_project_node(project_store, context_store)` — context_store 추가

In [ ]:
# 시그니처 변경: (project_store) → (project_store, context_store)
similar_project = make_similar_project_node(project_store, context_store)
similar_result = await similar_project(state)

print("=== similar_project 결과 ===")
similar_projects = similar_result.get("similar_projects", [])
if similar_projects:
    print(json.dumps(similar_projects, ensure_ascii=False, indent=2))
    for p in similar_projects:
        print(f"  ✓ {p.get('project_name')}: relevance={p.get('relevance_score')}, similarity={p.get('similarity_score')}")
else:
    print("유사 프로젝트 없음 (벡터 스토어에 데이터가 없는 경우 정상)")

In [ ]:
# Phase 1 결과 누적 — similar_projects
state.update(similar_result)

# phase1_sync 시뮬레이션 — Phase 1 완료, Phase 2 진입 전 state 확인
print("=" * 60)
print("=== Phase 1 완료 — phase1_sync 시뮬레이션 ===")
print("=" * 60)
print(f"현재 state keys: {list(state.keys())}")
print(f"resource_estimate 키: {list(state.get('resource_estimate', {}).keys())}")
print(f"similar_projects 수: {len(state.get('similar_projects', []))}")
print("\nPhase 2 노드(scoring, risk_analysis)는 이 state를 입력으로 받습니다.")
print("특히 resource_estimate가 있어야 scoring/risk_analysis가 정상 동작합니다.")

### 7. Phase 2: scoring 노드

Phase 2에서 risk_analysis와 병렬 실행됩니다.  
**핵심 변경**: state에 `resource_estimate`가 포함된 상태에서 실행 → 수익성/납기 평가 정확도 향상.  
**서버사이드 재계산**: LLM의 산술 오류를 방지하기 위해 `weighted_score`와 `total_score`를 서버에서 재계산합니다.

In [ ]:
scoring = make_scoring_node(context_store)
result = await scoring(state)  # state에 resource_estimate 포함

print("=== scoring 결과 ===")
print(f"total_score: {result.get('total_score')}")
print(f"verdict: {result.get('verdict')}")
print("\nscores (서버사이드 재계산 적용):")
for s in result.get("scores", []):
    print(f"  - {s['criterion']}: {s['score']}점 × {s['weight']} = {s['weighted_score']} — {s['rationale'][:60]}...")

# verdict 임계값 검증
ts = result.get("total_score", 0)
v = result.get("verdict", "")
if ts >= 70:
    assert v == "go", f"total_score={ts} ≥ 70이지만 verdict={v}"
elif ts >= 40:
    assert v == "conditional_go", f"total_score={ts} ≥ 40이지만 verdict={v}"
else:
    assert v == "no_go", f"total_score={ts} < 40이지만 verdict={v}"
print(f"\n✓ verdict 임계값 검증 통과: {ts}점 → {v}")

In [ ]:
# Phase 2 결과 누적 — scores, total_score, verdict
state.update(result)
print(f"현재 state keys: {list(state.keys())}")

### 8. Phase 2: risk_analysis 노드

Phase 2에서 scoring과 병렬 실행됩니다.  
**핵심 변경**: state에 `resource_estimate`가 포함된 상태에서 실행 → 재무/일정 리스크 구체화.  
**신규 출력**: `risk_interdependencies` (리스크 간 상호작용 분석)

In [ ]:
risk_analysis = make_risk_analysis_node(context_store)
result = await risk_analysis(state)  # state에 resource_estimate 포함

print("=== risk_analysis 결과 ===")
print("risks:")
for r in result.get("risks", []):
    print(f"  - [{r.get('level', 'N/A')}] {r.get('category', 'N/A')}: {r.get('item', '')} (확률: {r.get('probability')}, 영향: {r.get('impact')})")
    print(f"    {r.get('description', '')[:80]}...")

# risk_interdependencies 검증 (신규)
interdeps = result.get("risk_interdependencies", [])
print(f"\nrisk_interdependencies ({len(interdeps)}건):")
for ri in interdeps:
    print(f"  - {ri.get('risk_pair', [])} → {ri.get('combined_effect', '')}")

In [ ]:
# Phase 2 결과 누적 — risks, risk_interdependencies
state.update(result)
print(f"현재 state keys: {list(state.keys())}")

### 9. final_verdict 노드

In [ ]:
# 시그니처 변경: () → (context_store)
final_verdict = make_final_verdict_node(context_store)
result = await final_verdict(state)

print("=== final_verdict 결과 ===")
print(f"status: {result.get('status')}")
print(f"final_report 길이: {len(result.get('final_report', ''))} chars")

# 8개 섹션 존재 확인
report = result.get("final_report", "")
expected_sections = ["핵심 결론", "Deal 개요", "Deal 선정 기준 적합성", "평가 상세",
                     "예상 작업 범위", "리스크 분석", "유사 프로젝트", "권고사항"]
for section in expected_sections:
    status = "✓" if section in report else "✗"
    print(f"  {status} '{section}' 섹션")

print("\nfinal_report (처음 500자):")
print(report[:500], "...")

In [ ]:
# 최종 리포트 마크다운 렌더링
from IPython.display import Markdown, display

display(Markdown(result.get("final_report", "")))

### 10. 전체 파이프라인 state 요약

In [ ]:
state.update(result)

print("=== 최종 State 요약 ===")
print(f"deal_id: {state['deal_id']}")
print(f"status: {state.get('status')}")
print(f"verdict: {state.get('verdict')}")
print(f"total_score: {state.get('total_score')}")
print(f"scores count: {len(state.get('scores', []))}")
print(f"risks count: {len(state.get('risks', []))}")
print(f"risk_interdependencies count: {len(state.get('risk_interdependencies', []))}")
print(f"similar_projects count: {len(state.get('similar_projects', []))}")
print(f"resource_estimate keys: {list(state.get('resource_estimate', {}).keys())}")
print(f"final_report length: {len(state.get('final_report', ''))} chars")
print(f"errors: {state.get('errors', [])}")

# Phase 의존성 검증 요약
print("\n=== Phase 의존성 검증 ===")
has_re = bool(state.get("resource_estimate"))
has_sp = "similar_projects" in state
print(f"Phase 1 → Phase 2 의존성: resource_estimate={'✓' if has_re else '✗'}, similar_projects={'✓' if has_sp else '✗'}")